# Stage 5: Fine-tune DistilBERT on Colab (SWE015, Use Case 4)

**Before running:**
1. Menu **Runtime → Change runtime type → T4 GPU → Save**.
2. Left sidebar **🔑 Secrets** → **Add new secret** → Name `HF_TOKEN`, Value = your Hugging Face *write* token → switch **Notebook access** on.

Then run the cells **one by one, top to bottom** (click each cell, press **Shift+Enter**).

This notebook only *runs* `src/bert_train.py` from the GitHub repo. All the training logic is in that file.

## 1. Check that a GPU is attached
You should see a line like `GPU 0: Tesla T4`. If you see an error, redo step 1 above.

In [ ]:
!nvidia-smi -L

## 2. Get the code and install the pinned Hugging Face libraries
Colab already has PyTorch with GPU support, so we install only the Hugging Face libraries, at the same versions as `requirements.txt`.
If Colab shows **"Restart session"**, click it, then continue with cell 3 (don't rerun this one).

In [ ]:
%cd /content
!test -d swe015-chat-moderation || git clone https://github.com/mohdelsayedd/swe015-chat-moderation.git
%cd /content/swe015-chat-moderation
!git pull
!pip install -q transformers==5.24.0 accelerate==1.16.0 datasets==5.2.0 huggingface_hub==2.4.0

## 3. Log in to Hugging Face (reads the `HF_TOKEN` secret, the token is never printed)

In [ ]:
%cd /content/swe015-chat-moderation
from google.colab import userdata
from huggingface_hub import login, HfApi
login(token=userdata.get("HF_TOKEN").strip())  # strip(): a pasted token often carries a space/newline
HF_USER = HfApi().whoami()["name"]
REPO = f"{HF_USER}/swe015-distilbert-offensive"
print("Model will be uploaded to:", REPO)

## 4. Train (about 5–15 minutes on a T4)
This downloads the dataset and `distilbert-base-uncased`, fine-tunes for 3 epochs, keeps the epoch with the best **validation** macro-F1, and uploads the model to your Hugging Face account.

In [ ]:
!python -m src.bert_train --push {REPO}

## 5. Show the saved validation summary
Take a screenshot of this output and of the end of cell 4.

In [ ]:
!cat results/val_distilbert.json